# Interpretability: Which Input Features × Months Drive Each Embedding Dim?

Given a trained `TSEncoder` (76 numeric features × 24 months -> 64-dim
embedding), this notebook answers, for any chosen output dimension *d*:

1. **Which input features** are the embedding most sensitive to?
2. **Which months** matter most?
3. **Which feature × month *cells*** carry the joint signal?

Three complementary methods, none of which need labels:

| Method | What it answers | Cost |
|---|---|---|
| **Permutation importance** | per-feature (76,) | F forward passes per dim, model-agnostic |
| **Time-window masking** | per-month (24,) | T forward passes per dim, uses key_padding_mask |
| **Integrated Gradients** | per-(month, feature) heatmap (24, 76) | n_steps gradients, per dim |

"Importance" here means **sensitivity**: how much would `z_d` change if the
input were perturbed at that location. Strong sensitivity is the practical
proxy for "the embedding carries information from this input cell."

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader

from ts_embed.data import (DatasetPaths, TimeSeriesDataset,
                           TimeFeatureMasker, ContrastiveCollator)
from ts_embed.model import TSEncoder, TSEncoderConfig, TSEmbeddingModel
from ts_embed.loss import VICRegLoss, VICRegConfig

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Configuration

Point `CHECKPOINT_PATH` at your trained encoder. Pick a `DIM_OF_INTEREST`
(0..63). `FEATURE_NAMES` is a list of 76 strings used in the plots — replace
with your real feature names for readable charts.

In [ ]:
N, T, F_NUM, D_MODEL = 4000, 24, 76, 64
SAMPLE_SIZE = 1000                 # how many accounts to use for attribution
DIM_OF_INTEREST = 0                # change to investigate a different output dim
FEATURE_NAMES = [f'feat_{i:02d}' for i in range(F_NUM)]   # replace with real names

DATA_DIR        = REPO_ROOT / 'data_demo'
CHECKPOINT_PATH = REPO_ROOT / 'runs' / 'explainability' / 'encoder.pt'
print(f'embedding dims: {D_MODEL}  |  inputs: {F_NUM} features x {T} months')
print(f'investigating dim {DIM_OF_INTEREST}  |  sample size: {SAMPLE_SIZE}')

## 2. (Demo only) Bootstrap data + a trained encoder

Runs only if either the data files or the checkpoint are missing. Generates
synthetic 76-feature × 24-month data with latent structure, briefly pretrains
a TSEmbeddingModel with VICReg, and saves the encoder. **Skip if you have
your own files and checkpoint.**

In [ ]:
def _data_exists():
    return all((DATA_DIR / fn).exists() for fn in ['numeric.npy', 'missing.npy'])

if not _data_exists():
    print('generating synthetic demo data ...')
    rng = np.random.default_rng(0)
    # Inject latent structure: 4 blocks of 19 features each driven by independent
    # latent factors -- gives the encoder something to specialize on.
    n_blk = 4; per_blk = F_NUM // n_blk          # 19 per block
    factors = rng.standard_normal((N, n_blk)).astype(np.float32)
    numeric = 0.3 * rng.standard_normal((N, T, F_NUM)).astype(np.float32)
    t_axis = np.linspace(0, 1, T, dtype=np.float32)[None, :, None]
    for b in range(n_blk):
        sl = slice(b * per_blk, (b + 1) * per_blk)
        # block has its own trend pattern controlled by its latent factor
        numeric[:, :, sl] += factors[:, b, None, None] * (0.5 + t_axis)
    missing = (rng.random((N, T, F_NUM)) < 0.1).astype(np.uint8)
    fmean = numeric.mean((0, 1), keepdims=True)
    numeric = np.where(missing == 1, fmean, numeric).astype(np.float32)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    np.save(DATA_DIR / 'numeric.npy', numeric)
    np.save(DATA_DIR / 'missing.npy', missing)
    print(f'  wrote {DATA_DIR}/numeric.npy and missing.npy')
else:
    print(f'data files found at {DATA_DIR}')

if not CHECKPOINT_PATH.exists():
    print('no encoder checkpoint; running brief VICReg pretrain ...')
    paths_pre = DatasetPaths(numeric=DATA_DIR / 'numeric.npy',
                              missing=DATA_DIR / 'missing.npy',
                              categorical=None)
    pre_ds = TimeSeriesDataset(paths_pre, n_numeric=F_NUM)
    pre_masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30,
                                   n_time_spans=2, feat_span_frac=0.5)
    pre_loader = DataLoader(pre_ds, batch_size=256, shuffle=True, drop_last=True,
                            num_workers=0, collate_fn=ContrastiveCollator(pre_masker))
    pre_cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(), seq_len=T,
                              d_model=D_MODEL, n_layers=3, n_heads=4, proj_dim=64)
    pre_model = TSEmbeddingModel(pre_cfg).to(device)
    vicreg = VICRegLoss(VICRegConfig(gather_distributed=False))
    pre_opt = torch.optim.AdamW(pre_model.parameters(), lr=1e-3, weight_decay=1e-4)
    pre_model.train()
    for ep in range(5):
        for batch in pre_loader:
            num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
            keep_a = batch['time_keep_a'].to(device)
            num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
            keep_b = batch['time_keep_b'].to(device)
            _, z_a = pre_model(num_a, mis_a, None, keep_a)
            _, z_b = pre_model(num_b, mis_b, None, keep_b)
            loss = vicreg(z_a, z_b)['loss']
            pre_opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(pre_model.parameters(), 1.0); pre_opt.step()
        print(f'  vicreg epoch {ep}: loss={float(loss):.3f}')
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'model': pre_model.state_dict(), 'cfg': pre_cfg.__dict__}, CHECKPOINT_PATH)
    print(f'  saved demo encoder -> {CHECKPOINT_PATH}')
    del pre_model, pre_opt, pre_loader, pre_ds, vicreg, pre_masker
    if device.type == 'cuda': torch.cuda.empty_cache()
else:
    print(f'checkpoint found at {CHECKPOINT_PATH}')

## 3. Load the trained encoder + a sample batch

Handles `TSEmbeddingModel` (encoder.*+projector.*) or bare `TSEncoder`
checkpoints, plus DDP / torch.compile prefixes. The sample batch is what
we'll perturb to measure attribution.

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
cfg = TSEncoderConfig(**ckpt['cfg'])
encoder = TSEncoder(cfg).to(device)

state = {k.removeprefix('_orig_mod.').removeprefix('module.'): v
         for k, v in ckpt['model'].items()}
enc_state = {k[len('encoder.'):]: v for k, v in state.items()
             if k.startswith('encoder.')}
if not enc_state:
    enc_state = {k: v for k, v in state.items() if not k.startswith('projector.')}
encoder.load_state_dict(enc_state, strict=True)
encoder.eval()
print(f'encoder loaded; d_model={cfg.d_model}, n_numeric={cfg.n_numeric}, '
      f'cat_cardinalities={cfg.cat_cardinalities}')

# Sample batch from disk.
numeric_np = np.load(DATA_DIR / 'numeric.npy', mmap_mode='r')[:SAMPLE_SIZE]
missing_np = np.load(DATA_DIR / 'missing.npy', mmap_mode='r')[:SAMPLE_SIZE]
numeric = torch.from_numpy(np.asarray(numeric_np, dtype=np.float32)).to(device)
missing = torch.from_numpy(np.asarray(missing_np, dtype=np.float32)).to(device)

categorical = None
if cfg.cat_cardinalities and (DATA_DIR / 'categorical.npy').exists():
    cat_np = np.load(DATA_DIR / 'categorical.npy', mmap_mode='r')[:SAMPLE_SIZE]
    categorical = torch.from_numpy(np.asarray(cat_np, dtype=np.int64)).to(device)
    print(f'categorical shape: {tuple(categorical.shape)}')

with torch.no_grad():
    z_base = encoder(numeric, missing, categorical)
print(f'baseline embedding shape: {tuple(z_base.shape)}  (B, d_model)')

## 4. Three attribution methods

Helpers that return numpy arrays so the visualization downstream is easy.

In [ ]:
@torch.no_grad()
def permutation_feature_importance(encoder, numeric, missing, categorical=None,
                                    n_perms=3, seed=0):
    """For each feature f, shuffle that feature across the batch (preserving the
    customer's temporal pattern within that feature) and measure the mean
    |delta| in every output dim. Returns (D, F)."""
    encoder.eval()
    B, T_, F_ = numeric.shape
    gen = torch.Generator(device=numeric.device).manual_seed(seed)
    z_base = encoder(numeric, missing, categorical)
    D = z_base.size(1)
    imp = torch.zeros(D, F_, device=numeric.device)
    for f in range(F_):
        for _ in range(n_perms):
            num_perm = numeric.clone()
            perm = torch.randperm(B, generator=gen, device=numeric.device)
            num_perm[:, :, f] = numeric[perm, :, f]
            z_perm = encoder(num_perm, missing, categorical)
            imp[:, f] += (z_base - z_perm).abs().mean(0)
        imp[:, f] /= n_perms
    return imp.cpu().numpy()


@torch.no_grad()
def time_window_importance(encoder, numeric, missing, categorical=None):
    """For each month t, mask that month via the encoder's time_keep_mask and
    measure mean |delta| in every output dim. Returns (D, T)."""
    encoder.eval()
    B, T_, F_ = numeric.shape
    z_base = encoder(numeric, missing, categorical)
    D = z_base.size(1)
    imp = torch.zeros(D, T_)
    for t in range(T_):
        keep = torch.ones(B, T_, device=numeric.device)
        keep[:, t] = 0.0
        z_masked = encoder(numeric, missing, categorical, keep)
        imp[:, t] = (z_base - z_masked).abs().mean(0).cpu()
    return imp.numpy()


def integrated_gradients(encoder, numeric, missing, categorical=None, dim_d=0,
                          baseline=None, n_steps=20):
    """Integrated Gradients for one embedding dim.

    IG(x_{i,t,f}) = (x_{i,t,f} - x'_{i,t,f}) * mean over alpha of
                    d z_d(x' + alpha (x - x')) / d x_{i,t,f}
    
    Returns per-sample attribution of shape (B, T, F). A common baseline is
    zeros; per-feature mean is also reasonable.
    """
    encoder.eval()
    if baseline is None:
        baseline = torch.zeros_like(numeric)
    attr = torch.zeros_like(numeric)
    for step in range(n_steps + 1):
        alpha = step / n_steps
        interp = (baseline + alpha * (numeric - baseline)).detach().requires_grad_(True)
        z = encoder(interp, missing, categorical)
        grad = torch.autograd.grad(z[:, dim_d].sum(), interp)[0]
        attr = attr + grad
    attr = attr / (n_steps + 1)
    return ((numeric - baseline) * attr.detach()).cpu().numpy()

print('attribution helpers defined')

## 5. Attribution for `DIM_OF_INTEREST`

Run all three methods. The first two return matrices over **all** output
dims at once (we slice to the dim of interest); IG is per-dim so we call it
with `dim_d=DIM_OF_INTEREST`. Both per-feature (F=76) and per-month (T=24)
are quick. IG with 20 steps is the main cost.

In [ ]:
d = DIM_OF_INTEREST

print('computing permutation feature importance (all dims, 1 pass) ...')
perm_imp = permutation_feature_importance(encoder, numeric, missing, categorical, n_perms=3)
print('  shape:', perm_imp.shape)

print('computing time-window importance (all dims, 1 pass) ...')
time_imp = time_window_importance(encoder, numeric, missing, categorical)
print('  shape:', time_imp.shape)

print(f'computing integrated gradients for dim {d} ...')
ig = integrated_gradients(encoder, numeric, missing, categorical,
                          dim_d=d, n_steps=20)
ig_abs_mean = np.abs(ig).mean(axis=0)               # (T, F) heatmap for dim d
print('  IG attribution map shape:', ig_abs_mean.shape)

# Slice to dim of interest.
perm_d = perm_imp[d]                                 # (F,)
time_d = time_imp[d]                                 # (T,)

print(f'\n--- summary for embedding dim {d} ---')
TOP_K = 10
top_feat = np.argsort(perm_d)[-TOP_K:][::-1]
top_month = np.argsort(time_d)[-5:][::-1]
print(f'top {TOP_K} features (by permutation importance):')
for rank, f in enumerate(top_feat, 1):
    print(f'  {rank:2d}. {FEATURE_NAMES[f]:18s}  importance = {perm_d[f]:.4f}')
print(f'\ntop 5 months (by time-masking importance):')
for rank, t in enumerate(top_month, 1):
    print(f'  {rank}. month {t:2d}  importance = {time_d[t]:.4f}')

## 6. Visualize for `DIM_OF_INTEREST`

Three plots:

1. **IG heatmap** `(T, F)` — joint feature × month attribution.
2. **Top-K features** bar chart — permutation importance ranking.
3. **Per-month importance** line — recency vs older months.

In [ ]:
try:
    import matplotlib.pyplot as plt
    TOP_K = 15

    fig = plt.figure(figsize=(18, 5.5))
    gs = fig.add_gridspec(1, 3, width_ratios=[2.2, 1.4, 1.4], wspace=0.35)

    # 1. IG heatmap (T x F) -- months on Y, features on X.
    ax0 = fig.add_subplot(gs[0, 0])
    im = ax0.imshow(ig_abs_mean, aspect='auto', cmap='viridis',
                    interpolation='nearest')
    ax0.set_xlabel('feature index'); ax0.set_ylabel('month')
    ax0.set_title(f'|Integrated Gradients| for dim {d}\n(mean over {SAMPLE_SIZE} samples)')
    ax0.set_yticks(range(0, T, 2))
    plt.colorbar(im, ax=ax0, fraction=0.04)

    # 2. Top-K features by permutation importance, bar chart.
    ax1 = fig.add_subplot(gs[0, 1])
    top_f = np.argsort(perm_d)[-TOP_K:]              # ascending for bar plot
    ax1.barh(range(TOP_K), perm_d[top_f], color='steelblue')
    ax1.set_yticks(range(TOP_K))
    ax1.set_yticklabels([FEATURE_NAMES[f] for f in top_f], fontsize=8)
    ax1.set_xlabel('mean |Δz_d|')
    ax1.set_title(f'top-{TOP_K} features for dim {d}\n(permutation importance)')
    ax1.grid(alpha=0.3, axis='x')

    # 3. Per-month importance, line chart.
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.plot(range(T), time_d, marker='o', linewidth=2, color='darkorange')
    ax2.fill_between(range(T), 0, time_d, alpha=0.2, color='darkorange')
    ax2.set_xlabel('month'); ax2.set_ylabel('mean |Δz_d| when month masked')
    ax2.set_title(f'per-month importance for dim {d}\n(time-window masking)')
    ax2.set_xticks(range(0, T, 2))
    ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed; skipping plots')

## 7. Attribution across all 64 embedding dims

`perm_imp` is `(64, 76)` — *which features* drive *each* embedding dim.
`time_imp` is `(64, 24)` — *which months* drive each dim. Two heatmaps plus a
compact table listing the top-3 features per dim.

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    im1 = axes[0].imshow(perm_imp, aspect='auto', cmap='viridis')
    axes[0].set_xlabel('feature index'); axes[0].set_ylabel('embedding dim')
    axes[0].set_title('per-feature importance for every output dim\n(permutation)')
    plt.colorbar(im1, ax=axes[0], fraction=0.04)
    # mark the row corresponding to DIM_OF_INTEREST
    axes[0].axhline(d, color='red', linewidth=0.7, alpha=0.6)

    im2 = axes[1].imshow(time_imp, aspect='auto', cmap='viridis')
    axes[1].set_xlabel('month'); axes[1].set_ylabel('embedding dim')
    axes[1].set_title('per-month importance for every output dim\n(time-window masking)')
    plt.colorbar(im2, ax=axes[1], fraction=0.04)
    axes[1].axhline(d, color='red', linewidth=0.7, alpha=0.6)

    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed; skipping global heatmaps')

# Compact per-dim top-3 table.
print('\ntop-3 features per embedding dim (perm importance):')
print(f'{"dim":>4} | {"#1":<28} | {"#2":<28} | {"#3":<28}')
print('-' * 100)
for dd in range(D_MODEL):
    top3 = np.argsort(perm_imp[dd])[-3:][::-1]
    cells = ' | '.join(f'{FEATURE_NAMES[f][:18]:18s} ({perm_imp[dd, f]:.3f})' for f in top3)
    print(f'{dd:>4} | {cells}')

## Next steps

- **Per-cell permutation** — `permutation_feature_importance` shuffles a
  whole feature's column. For a finer (T, F) cell-level view, shuffle only
  the (t, f) entry; this is `T x F` forward passes per dim (slower but
  directly comparable to the IG heatmap).
- **Different IG baseline** — zero isn't always meaningful for normalized
  features. Try the per-feature batch mean, or a particular customer's
  values you want to explain *relative to*.
- **Smoothing** — IG can be noisy on individual cells; either average over
  more samples or add Gaussian-noise replicates ('SmoothGrad-IG').
- **Per-customer attribution** — IG is already per-sample (`ig` has shape
  `(B, T, F)`). Slice to one customer and plot their personal heatmap when
  you need a case-level explanation.
- **Categorical features** — gradient-based attribution doesn't apply
  directly to integer-indexed embeddings. Use **permutation** (shuffle the
  categorical column across the batch) instead.
- **Aspect interpretation** — if your encoder uses partitioned chunks
  (e.g. `StructuredContrastiveLoss` / `AspectAugContrastiveLoss`), pick a
  representative dim from each chunk's slice and check that its top features
  match that chunk's semantic — a clean sanity test of specialization.